### Config

In [0]:
cards_df = spark.table("jrvs_databricks_fundamentals.silver.cards_data")
transactions_df = spark.table("jrvs_databricks_fundamentals.silver.transactions_data")
user_df = spark.table("jrvs_databricks_fundamentals.silver.users_data")
mcc_df = spark.table("jrvs_databricks_fundamentals.silver.mcc_codes")
fraud_df = spark.table("jrvs_databricks_fundamentals.silver.fraud_labels")

# Fact table

In [0]:
from pyspark.sql.functions import (
    col, to_date, date_format, hour,
    weekofyear, year, concat_ws, when, lit, avg, min as _min, round as _round,
)
from pyspark.sql.window import Window

# 1. Build time/date columns
fact = (
    fact
    .withColumn("date_only",   to_date("date"))
    .withColumn("day_of_week", date_format("date", "EEEE"))
    .withColumn("hour",        hour("date"))
    .withColumn("year_week",   concat_ws("-", year("date"), weekofyear("date")))
    .withColumn("year_month",  date_format("date", "yyyy-MM"))
    .withColumn("time_of_day", date_format("date", "H:mm"))               # clock time (detail)
    .withColumn("day_period",                                             # Q7 morning vs night
        when((col("hour") >= 6) & (col("hour") < 18), lit("morning")).otherwise(lit("night")))
    .withColumn("amount_band",                                            # Q14 high vs low
        when(col("amount") >= 100, lit("high")).otherwise(lit("low")))
)

# 2. Q4, Q13 window columns
user_week = Window.partitionBy("user_id", "year_week")
user_all  = Window.partitionBy("user_id")

fact = (
    fact
    # Q4
    .withColumn("user_weekly_avg_amount", avg("amount").over(user_week))
    .withColumn("amount_vs_weekly_avg",
        when(col("user_weekly_avg_amount") > 0,
             col("amount") / col("user_weekly_avg_amount")).otherwise(lit(None)))
    .withColumn("is_amount_spike", col("amount_vs_weekly_avg") >= 3)      # 3x = spike
    # Q13
    .withColumn("first_fraud_date",
        _min(when(col("is_fraud") == True, col("date_only"))).over(user_all))
    .withColumn("is_post_fraud",
        when(col("first_fraud_date").isNull(), lit(False))
            .otherwise(col("date_only") > col("first_fraud_date")))
)

# 3. Format window columns
fact = (
    fact
    .withColumn("user_weekly_avg_amount", col("user_weekly_avg_amount").cast("decimal(19,2)"))  
    .withColumn("amount_vs_weekly_avg", _round("amount_vs_weekly_avg", 2))                      
)

# Reorder columns for logitical reason
fact = fact.select(
    "transaction_id", "user_id", "card_id", "merchant_id",
    "date", "amount", "is_fraud",
    "card_type", "card_brand",
    "mcc", "mcc_description",
    "date_only", "day_of_week", "hour", "time_of_day", "day_period", "year_week", "year_month",
    "amount_band", "user_weekly_avg_amount", "amount_vs_weekly_avg", "is_amount_spike",   # Q4 / Q14
    "first_fraud_date", "is_post_fraud",                                                   # Q13
)


(fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.fact_transactions"))

fact.printSchema()
display(fact.limit(10))

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- amount: decimal(19,2) (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- mcc: short (nullable = true)
 |-- mcc_description: string (nullable = true)
 |-- date_only: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- day_period: string (nullable = false)
 |-- year_week: string (nullable = false)
 |-- year_month: string (nullable = true)
 |-- amount_band: string (nullable = false)
 |-- user_weekly_avg_amount: decimal(19,2) (nullable = true)
 |-- amount_vs_weekly_avg: decimal(26,2) (nullable = true)
 |-- is_amount_spike: boolean (nullable = true)
 |-- first_fraud_date: date (nullable 

transaction_id,user_id,card_id,merchant_id,date,amount,is_fraud,card_type,card_brand,mcc,mcc_description,date_only,day_of_week,hour,time_of_day,day_period,year_week,year_month,amount_band,user_weekly_avg_amount,amount_vs_weekly_avg,is_amount_spike,first_fraud_date,is_post_fraud
7489404,3,2157,91290,2010-01-04T14:02:00.000Z,54.57,null,Debit (Prepaid),Visa,5300,Wholesale Clubs,2010-01-04,Monday,14,14:02,morning,2010-1,2010-01,low,43.24,1.26,false,2013-10-18,false
7489608,3,2157,81833,2010-01-04T14:51:00.000Z,55.85,false,Debit (Prepaid),Visa,5912,Drug Stores and Pharmacies,2010-01-04,Monday,14,14:51,morning,2010-1,2010-01,low,43.24,1.29,false,2013-10-18,false
7490494,3,2157,19964,2010-01-04T19:24:00.000Z,0.86,null,Debit (Prepaid),Visa,5311,Department Stores,2010-01-04,Monday,19,19:24,night,2010-1,2010-01,low,43.24,0.02,false,2013-10-18,false
7493590,3,2157,81833,2010-01-05T14:59:00.000Z,49.88,false,Debit (Prepaid),Visa,5912,Drug Stores and Pharmacies,2010-01-05,Tuesday,14,14:59,morning,2010-1,2010-01,low,43.24,1.15,false,2013-10-18,false
7497616,3,2157,78680,2010-01-06T14:59:00.000Z,54.26,false,Debit (Prepaid),Visa,5411,"Grocery Stores, Supermarkets",2010-01-06,Wednesday,14,14:59,morning,2010-1,2010-01,low,43.24,1.25,false,2013-10-18,false
7501495,3,2157,19964,2010-01-07T14:40:00.000Z,59.79,null,Debit (Prepaid),Visa,5311,Department Stores,2010-01-07,Thursday,14,14:40,morning,2010-1,2010-01,low,43.24,1.38,false,2013-10-18,false
7508209,3,2157,81833,2010-01-09T12:10:00.000Z,7.85,false,Debit (Prepaid),Visa,5912,Drug Stores and Pharmacies,2010-01-09,Saturday,12,12:10,morning,2010-1,2010-01,low,43.24,0.18,false,2013-10-18,false
7508773,3,2157,61195,2010-01-09T14:33:00.000Z,59.67,null,Debit (Prepaid),Visa,5541,Service Stations,2010-01-09,Saturday,14,14:33,morning,2010-1,2010-01,low,43.24,1.38,false,2013-10-18,false
7512673,3,2157,19964,2010-01-10T13:52:00.000Z,46.47,false,Debit (Prepaid),Visa,5311,Department Stores,2010-01-10,Sunday,13,13:52,morning,2010-1,2010-01,low,43.24,1.07,false,2013-10-18,false
7737295,3,2157,51180,2010-03-08T10:32:00.000Z,37.83,null,Debit (Prepaid),Visa,5193,"Florists Supplies, Nursery Stock and Flowers",2010-03-08,Monday,10,10:32,morning,2010-10,2010-03,low,38.83,0.97,false,2013-10-18,false
